In [1]:
from pydantic import BaseModel, Field
import json
from pathlib import Path

from dotenv import load_dotenv
import dspy

In [2]:
class DraftSyntheticEmailDescription(dspy.Signature):
    """Given a ClinicalTrials.gov study ID and a role (eg CRA, CTM), create descriptions of email threads that the person with the role (at the sponsor level) would be expected to be help resolve.
    Vary the email threads to cover different types of common clinical trial communications (eg SAE reporting, monitoring visit scheduling, data queries, protocol deviations, etc).
    Vary the urgency of the email threads (eg routine, urgent, critical).
    Vary the complexity of the email threads (eg single email, multi-email back and forth).
    Vary the source of the email thread (site to sponsor, CRO to sponsor, sponsor to site, etc).

    Ex for a CRA, make one email thread about scheduling a monitoring visit, one about following up on overdue data queries, one about reporting an SAE, etc.
    """

    study_overview: str = dspy.InputField(desc="The ClinicalTrials.gov study overview")
    role: str = dspy.InputField(
        desc="The role of the person who would be expected to take action in response to the email thread"
    )
    organization: str = dspy.InputField(
        desc="The organization of the person with the given role (eg sponsor, CRO, site)"
    )
    num_threads: int = dspy.InputField(
        desc="The number of email threads to create descriptions for"
    )

    # Outputs
    email_descriptions: list[str] = dspy.OutputField(
        desc="A list of descriptions of the email threads that the role would be expected to help resolve. Don't number threads."
    )


class Email(BaseModel):
    """An email message including metadata for clinical trial communications.

    Metadata fields:
    - from_address: Email address of the sender (e.g., site coordinator, CRA, medical monitor)
    - to_addresses: List of primary recipient email addresses who need to take action
    - cc_addresses: List of email addresses copied for awareness (e.g., project managers, medical monitors)
    - subject: Email subject line, often includes study ID, urgency indicators, and topic
    - timestamp: ISO 8601 formatted timestamp indicating when the email was sent
    - body: Full text content of the email message, including greetings, content, and signature
    - attachments: List of attachment filenames (e.g., SAE forms, source documents, meeting agendas)
    - message_id: Unique identifier for the email message in the thread
    - in_reply_to: Message ID of the email being replied to, or None if this starts a new thread
    """

    from_address: str
    to_addresses: list[str]
    cc_addresses: list[str]
    subject: str
    timestamp: str
    body: str
    attachments: list[str]
    message_id: int
    in_reply_to: int | None


class DraftSyntheticEmailThread(dspy.Signature):
    """Given a description of an email thread and a role of a person to include, draft a synthetic email thread that includes the person with the given role (this person is always at the sponsor organization).
    Come up with names for organizations (sponsor, CRO, site) people, attachments in the email thread as needed to make the email thread realistic.
    """

    email_description: str = dspy.InputField(desc="The description of the email thread to draft")
    role: str = dspy.InputField(
        desc="The role of the person (at the sponsor level) that must take action (reply to email, redirect email, confirm follow up, schedule meeting, etc) in the email thread. This person is always at the sponsor organization."
    )
    organization: str = dspy.InputField(
        desc="The organization of the person with the given role (eg sponsor, CRO, site) that must take action in the email thread."
    )

    email_thread: list[Email] = dspy.OutputField(desc="The synthetic email thread")
    role_descriptions: list[str] = dspy.OutputField(
        desc="""Map of "email addresses: organization, role" in the email thread. 
        Organization can be sponsor, CRO or site. 
        Given role is always at sponsor organization and must be included in mapping.
        Ex: jane.doe@denalitx.com: Sponsor, CRA"""
    )

In [3]:
load_dotenv("../.env")
lm = dspy.LM("gemini/gemini-2.5-pro", temperature=0.5, cache=True, max_tokens=25000)
dspy.settings.configure(lm=lm, track_usage=True)

In [4]:
STUDY_OVERVIEW = """
            Brief Summary

            This is a Phase 1/2, multicenter, randomized, placebo-controlled, double-blind study to evaluate the safety, tolerability, pharmacokinetics (PK), and pharmacodynamics (PD) of single and multiple doses of DNL593 in two parts followed by an optional open-label extension (OLE) period.

            Part A will evaluate the safety, tolerability, PK, and PD of single doses of DNL593 in healthy male and healthy female participants of nonchildbearing potential. Part B will evaluate the safety, tolerability, PK, and PD of multiple doses of DNL593 in participants with frontotemporal dementia (FTD) over 25 weeks. Part B will be followed by Part C, an optional 18-month OLE period available for all participants who complete Part B.
            Official Title
            A Phase 1/2, Multicenter, Randomized, Placebo-Controlled, Double Blind Single Dose and Multiple Dose Study to Evaluate the Safety, Tolerability, Pharmacokinetics, and Pharmacodynamics of DNL593 in Healthy Participants and Participants With Frontotemporal Dementia Followed by an Open-Label Extension
            Conditions
            Frontotemporal Dementia
            Intervention / Treatment

                Drug: DNL593
                Drug: Placebo

            Sponsor: Denali Therapeutics
          """

ROLE = "Clinical Research Associate"

ORGANIZATION = "Sponsor"

NUM_THREADS = 10

In [5]:
email_descriptions_predict = dspy.Predict(DraftSyntheticEmailDescription)
email_descriptions = email_descriptions_predict(
    study_overview=STUDY_OVERVIEW, role=ROLE, organization=ORGANIZATION, num_threads=NUM_THREADS
).email_descriptions

email_descriptions

["An urgent email thread from the Principal Investigator at a participating site reporting a Serious Adverse Event (SAE). A patient in Part B (FTD cohort) has been hospitalized for a severe seizure. The thread includes the initial notification, the CRA's immediate request for the SAE form and source documents, and the CRA's subsequent communication with the internal Denali safety team.",
 'A routine back-and-forth email exchange between the CRA and a Site Coordinator to schedule an upcoming interim monitoring visit. The discussion covers potential dates, confirmation of the agenda (including 100% SDV for the first two FTD patients), and a request to have specific patient binders and ISF documents available for review.',
 "A follow-up email from the CRA to a site's data manager regarding a list of critical data queries that are now over 14 days past due. The CRA emphasizes the importance of resolving these queries ahead of the upcoming database lock for the Part A (healthy volunteer) an

In [ ]:
threads = []
output_dir = Path("generated_email_threads")
output_dir.mkdir(exist_ok=True, parents=True)

for idx, description in enumerate(email_descriptions):
    print("Generating email thread for description: ", description)

    email_thread_predict = dspy.Predict(DraftSyntheticEmailThread)
    email_thread_result = email_thread_predict(
        email_description=description, role=ROLE, organization=ORGANIZATION
    )

    emails_data = [
        {
            "from_address": email.from_address,
            "to_addresses": email.to_addresses,
            "cc_addresses": email.cc_addresses,
            "subject": email.subject,
            "timestamp": email.timestamp,
            "body": email.body,
            "attachments": email.attachments,
            "message_id": email.message_id,
            "in_reply_to": email.in_reply_to,
        }
        for email in email_thread_result.email_thread
    ]

    thread_data = {
        "emails": emails_data,
        "role_descriptions": email_thread_result.role_descriptions,
        "description": description,
    }
    threads.append(thread_data)

    thread_file = output_dir / f"thread_{idx:03d}.json"
    with open(thread_file, "w") as f:
        json.dump(thread_data, f, indent=2)
    print(f"Saved thread {idx} to {thread_file}")

summary_file = output_dir / "all_threads.json"
with open(summary_file, "w") as f:
    json.dump({"study_overview": STUDY_OVERVIEW, "role": ROLE, "threads": threads}, f, indent=2)
print(f"Saved summary to {summary_file}")

Generating email thread for description:  An urgent email thread from the Principal Investigator at a participating site reporting a Serious Adverse Event (SAE). A patient in Part B (FTD cohort) has been hospitalized for a severe seizure. The thread includes the initial notification, the CRA's immediate request for the SAE form and source documents, and the CRA's subsequent communication with the internal Denali safety team.
Saved thread 0 to generated_email_threads_2/thread_000.json
Generating email thread for description:  A routine back-and-forth email exchange between the CRA and a Site Coordinator to schedule an upcoming interim monitoring visit. The discussion covers potential dates, confirmation of the agenda (including 100% SDV for the first two FTD patients), and a request to have specific patient binders and ISF documents available for review.
Saved thread 1 to generated_email_threads_2/thread_001.json
Generating email thread for description:  A follow-up email from the CRA t